In [3]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install faiss-cpu sentence-transformers numpy pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.0 MB/s eta 0:00:00


In [4]:
import os
import pickle
import numpy as np
import faiss

In [5]:
# Uncomment if using Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
BASE_PATH = "/content/drive/MyDrive/database"

In [7]:
embedding_path = f"{BASE_PATH}/embeddings.npy"

embeddings = np.load(embedding_path)

print("Embeddings Shape:", embeddings.shape)

Embeddings Shape: (50, 384)


In [8]:
metadata_path = f"{BASE_PATH}/metadata.pkl"

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Metadata Records:", len(metadata))

Metadata Records: 50


In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Added :", index.ntotal)

Vectors Added : 50


In [10]:
faiss_path = f"{BASE_PATH}/faiss.index"

faiss.write_index(index, faiss_path)

print("FAISS Index Saved")

FAISS Index Saved


In [11]:
print(os.listdir(BASE_PATH))

['persons.xlsx', 'criminals.xlsx', 'officers.xlsx', 'evidence.xlsx', 'fir.xlsx', 'wanted.xlsx', 'cases.xlsx', 'charges.xlsx', 'arrests.xlsx', 'convictions.xlsx', 'warrants.xlsx', 'users.xlsx', 'merged_police_data.xlsx', 'documents.pkl', 'embeddings.npy', 'metadata.pkl', 'faiss.index']


In [12]:
index = faiss.read_index(f"{BASE_PATH}/faiss.index")

print("Loaded Index:", index.ntotal)

Loaded Index: 50


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
def search(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    return scores, indices

In [15]:
scores, indices = search("Vehicle theft in Chennai")

print(scores)
print(indices)

[[0.79815817 0.78868073 0.78705174 0.7835943  0.78341573]]
[[40 20 15  0  5]]


In [17]:
for idx in indices[0]:

    print("-"*80)

    print(metadata[idx])

    print()

--------------------------------------------------------------------------------
{'fir_number': 'TNFIR20260041', 'case_number': 'CASE041', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Ashok', 'officer': 'R.Kumar 41', 'rank': 'Inspector', 'status': 'Under Investigation'}

--------------------------------------------------------------------------------
{'fir_number': 'TNFIR20260021', 'case_number': 'CASE021', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Bala', 'officer': 'R.Kumar 21', 'rank': 'Inspector', 'status': 'Under Investigation'}

--------------------------------------------------------------------------------
{'fir_number': 'TNFIR20260016', 'case_number': 'CASE016', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Lakshmi', 'officer': 'K.Murugan 16', 'rank': 'Inspector', 'status': 'Under Investigation'}

----------------

In [18]:
documents_path = f"{BASE_PATH}/documents.pkl"

with open(documents_path, "rb") as f:
    documents = pickle.load(f)

In [19]:
for idx in indices[0]:

    print("="*100)

    print(documents[idx].page_content)

    print()



================ FIR DETAILS ================

FIR Number : TNFIR20260041

Case Number : CASE041

Crime Category : Theft

Crime Type : Vehicle Theft

IPC Sections : 379

Description :

Vehicle Theft reported at Chennai

Investigation Status :

Under Investigation

Charge Sheet :

Pending

Police Station :

T. Nagar Police Station

District :

Chennai

Incident Date :

2026-07-14

Incident Time :

09:45


================ PERSON DETAILS ================

Person Name :

Ashok

Gender :

Male

DOB :

1991-06-14

Occupation :

Teacher

Address :

No.41, Gandhi Street

Mobile :

9800000041


================ CRIMINAL DETAILS ================

Alias :

Alias-41

Gang :

Red Tigers

Risk Level :

High

Previous Cases :

5

Arrest Count :

2

Repeat Offender :

No

Bail Status :

Denied

Current Status :

Under Investigation

Wanted :

No


================ OFFICER DETAILS ================

Officer Name :

R.Kumar 41

Rank :

Inspector

Station :

T. Nagar Police Station

Officer District :
